
# Ημέρα 2 — Overfitting, Regularization & Threshold Tuning (Recall ≥ 0.90)

**Week 3 · cc-fraud-detection**  
**Στόχος:** Επιλογή έντασης L2 regularization (hyperparameter **C**) σε Logistic Regression και **threshold tuning** για **Recall ≥ 0.90** (ή άλλο επιχειρησιακό στόχο).  
**Συγγραφέας:** Lazaros Voulistiotis  
**Ημερομηνία δημιουργίας notebook:** 2025-09-14 17:33

---

### Τι θα κάνεις
- Φόρτωση dataset & στόχου (binary: 0/1)
- Grid σε **C ∈ {0.01, 0.1, 1, 10}**
- Αξιολόγηση σε **threshold=0.5** (Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC)
- Επιλογή **καλύτερου C** βάσει **PR-AUC (Average Precision)**
- **Threshold tuning** για να πετύχεις **Recall ≥ 0.90**
- Αποθήκευση γραφημάτων: ROC/PR ανά C, PR στο best C με σημείο threshold, και Confusion Matrix στο επιλεγμένο threshold

> 💡 **Συμβουλή:** Σε εντονό **class imbalance** προτίμησε PR-AUC για επιλογή μοντέλου και χρησιμοποίησε **class_weight="balanced"**.


In [1]:

# ---- ΒΙΒΛΙΟΘΗΚΕΣ ----
import numpy as np               # αριθμητικοί πίνακες, μαθηματικές πράξεις
import pandas as pd              # dataframes, φόρτωση/επεξεργασία δεδομένων
import matplotlib.pyplot as plt  # γραφήματα, visualization

# --- scikit-learn modules ---
from sklearn.model_selection import train_test_split
# Χωρίζει dataset σε train/test (με stratify, random_state κλπ.)

from sklearn.preprocessing import StandardScaler
# Standardization (μέσος όρος=0, std=1) — απαραίτητο για Logistic Regression με L2 regularization

from sklearn.pipeline import Pipeline
# Επιτρέπει να φτιάξεις ένα pipeline: π.χ. StandardScaler -> LogisticRegression

from sklearn.linear_model import LogisticRegression
# Ο classifier μας: Logistic Regression με παραμέτρους (C, penalty, class_weight κ.λπ.)

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)
# Accuracy = ποσοστό σωστών προβλέψεων
# Precision = από όλα τα "fraud" που βρήκε, πόσα ήταν όντως fraud
# Recall = από όλα τα πραγματικά fraud, πόσα βρήκε
# F1 = ισορροπία Precision/Recall
# ROC-AUC = συνολική ικανότητα διάκρισης (0.5=τυχαίο, 1=τέλειο)
# Average Precision (PR-AUC) = area κάτω από Precision–Recall curve (σημαντικό σε imbalanced data)
# roc_curve, precision_recall_curve = για καμπύλες thresholds
# confusion_matrix = πίνακας σφαλμάτων (TP, FP, FN, TN)
# classification_report = summary με precision, recall, f1 ανά κλάση

In [2]:

# ---- ΡΥΘΜΙΣΕΙΣ ----
from pathlib import Path         # για cross-platform paths (αντί για raw strings "C:/...")

# Πού βρίσκεται το dataset. Χρησιμοποιούμε Path για cross-platform συμβατότητα.
# Αν αλλάξει η δομή φακέλων ή θες άλλο dataset, αλλάζεις ΜΟΝΟ εδώ.
DATA_PATH = Path("../data/data_raw/creditcard.csv")  # προσαρμόσ’ το αν χρειάζεται

# Πιθανά ονόματα της στήλης-στόχου (binary labels 0/1).
# Σε διάφορα datasets η στήλη μπορεί να λέγεται "Class", "class" ή "is_fraud".
TARGET_CANDIDATES = ["Class", "class", "is_fraud"]

# Τιμές για την υπερπαράμετρο C της Logistic Regression (L2). 
# Θυμήσου: μικρό C = πιο ισχυρή regularization, μεγάλο C = πιο “χαλαρή”.
# Τα έχουμε σε λογαριθμική κλίμακα (0.01 → 0.1 → 1 → 10) για να καλύψουμε ευρύ φάσμα.
CS = [0.01, 0.1, 1, 10]  # grid σε C

# Αναλογία test set. 0.30 = 30% test, 70% train. 
# Σε imbalanced data, το 30% βοηθά να έχεις αρκετά θετικά (fraud) στο test.
TEST_SIZE = 0.30

# Σπόρος τυχαιότητας για reproducibility.
# Με ίδιο RANDOM_STATE παίρνεις ίδιο split/αποτελέσματα σε κάθε run (σημαντικό για repo/reports).
RANDOM_STATE = 42

# Στόχος για tuning του threshold: επιδιώκεις τουλάχιστον αυτό το recall.
# Προσοχή στο trade-off: όσο ανεβάζεις στόχο recall, τόσο πιθανότερα πέφτει το precision (περισσότερα false positives).
RECALL_TARGET = 0.90  # στόχος threshold tuning



In [3]:

# ---- HELPERS ----

def find_target(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Δεν βρήκα target σε {candidates}. Δώσε σωστή στήλη στόχου (0/1).")


- Σκοπός: βρίσκει ποια στήλη του DataFrame είναι το target (0/1).
- Λογική: ψάχνει μέσα στις πιθανές (candidates = π.χ. ["Class", "class", "is_fraud"]) και επιστρέφει την πρώτη που βρει.
- Fail case: αν δεν βρει καμία, σηκώνει ValueError με κατανοητό μήνυμα.
💡 Καλή πρακτική γιατί datasets μπορεί να έχουν διαφορετικά ονόματα στηλών.

In [4]:
def ensure_outdir():
    """
    Δημιουργεί (αν δεν υπάρχει ήδη) τον φάκελο εξόδου για τα γραφήματα.
    Επιστρέφει:
        Path: Το αντικείμενο Path που δείχνει στον φάκελο 'images/week3'.
    """
    out = Path("images/week3")                 # Ο φάκελος όπου θα αποθηκευτούν όλα τα plots
    out.mkdir(parents=True, exist_ok=True)     # Δημιούργησε τον φάκελο και τους γονικούς (αν χρειάζεται)
                                               # exist_ok=True: δεν πετάει error αν υπάρχει ήδη
    return out                                 # Επιστροφή του Path ώστε να χρησιμοποιηθεί σε plt.savefig()


In [13]:
def print_metrics_table(rows, title="Metrics per C"):
    """
    Εμφανίζει έναν πίνακα με τις μετρικές από το πείραμα Logistic Regression.
    Χρησιμοποιεί Styler API (pandas >= 1.4).
    """
    dfm = pd.DataFrame(rows)
    display(dfm.style.hide(axis="index").set_caption(title))


In [15]:
def choose_threshold_by_recall(y_true, proba, recall_target=0.90):
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    candidates = []
    for i in range(len(thresholds)):
        r = recall[i]
        p = precision[i]
        thr = thresholds[i]
        if r >= recall_target:
            candidates.append((thr, p, r))
# Σκοπός: βρίσκει το μικρότερο threshold που πιάνει recall ≥ στόχο.
# precision_recall_curve επιστρέφει τριπλέτα (precision, recall, thresholds).
# Για κάθε threshold, κρατάς precision, recall, threshold.
# Αν το recall στο συγκεκριμένο threshold ≥ στόχος (π.χ. 0.90), τον βάζεις στους υποψήφιους (candidates).
    if not candidates:
        best_i = int(np.argmax(recall))
        best_thr = thresholds[min(best_i, len(thresholds)-1)] if len(thresholds) > 0 else 0.5
        return best_thr, float(precision[best_i]), float(recall[best_i])
# Case: Αν δεν υπάρχει threshold που να πιάνει recall ≥ στόχο.
# Τότε παίρνεις το μέγιστο recall διαθέσιμο (np.argmax(recall)).
# Αν δεν υπάρχουν thresholds (empty case), κάνεις fallback σε 0.5.
# Επιστρέφεις (threshold, precision, recall) του καλύτερου σημείου που κατάφερες να βρεις.

    # Αν υπάρχουν υποψήφιοι thresholds
    best_thr, best_p, best_r = max(candidates, key=lambda x: (x[1], -x[0]))
    return float(best_thr), float(best_p), float(best_r)
#Από τους υποψήφιους, διαλέγεις αυτόν με το υψηλότερο precision (για να περιορίσεις τα false positives), και αν υπάρχει ισοβαθμία, προτιμάς το μικρότερο threshold.
# Επιστρέφεις threshold, precision, recall ως floats (για καθαρό formatting).


🔑 Τι κάνει συνολικά αυτό το block

- find_target → ποια είναι η στήλη στόχος
- ensure_outdir → βεβαιώσου ότι υπάρχει φάκελος εικόνων
- print_metrics_table → εμφάνισε metrics σε Jupyter-friendly πίνακα
- choose_threshold_by_recall → διάλεξε threshold που πετυχαίνει recall ≥ στόχος με το καλύτερο δυνατό precision


## 2) Φόρτωση Dataset
- Αντί για `creditcard.csv` μπορείς να ορίσεις άλλο αρχείο/μονοπάτι.  
- Το target πρέπει να είναι μία στήλη **0/1** με όνομα ένα από: `Class`, `class`, `is_fraud` (ή άλλαξε το helper).


In [16]:
# ---- ΦΟΡΤΩΣΗ ΔΕΔΟΜΕΝΩΝ ----

# Έλεγχος: αν δεν υπάρχει το dataset στη διαδρομή που έχουμε ορίσει (DATA_PATH),
# σταμάτα το πρόγραμμα και εμφάνισε καθαρό μήνυμα σφάλματος.
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing {DATA_PATH}. Put the dataset in data/data_raw/")

# Βεβαιώσου ότι υπάρχει ο φάκελος για αποθήκευση γραφημάτων (images/week3).
# Αν δεν υπάρχει, τον δημιουργεί αυτόματα.
outdir = ensure_outdir()

# Φόρτωσε το dataset από το CSV σε pandas DataFrame.
df = pd.read_csv(DATA_PATH)

# Εντόπισε ποια στήλη είναι το target (π.χ. "Class" ή "is_fraud") 
# χρησιμοποιώντας τη helper συνάρτηση find_target.
y_col = find_target(df, TARGET_CANDIDATES)

# Μετατροπή της στήλης στόχου σε ακέραια (0 και 1).
y = df[y_col].astype(int)

# Features (X): αφαίρεσε τη στήλη στόχου και κράτησε μόνο αριθμητικά πεδία.
# Αυτό αποτρέπει σφάλματα αν το dataset περιέχει κατηγορικές στήλες ή κείμενα.
X = df.drop(columns=[y_col]).select_dtypes(include=[np.number])

# Εκτύπωσε βασικές πληροφορίες για έλεγχο:
# - To shape των X και y (πλήθος γραμμών και χαρακτηριστικών).
# - Το όνομα της target column.
print("Shape X, y:", X.shape, y.shape)
print("Target column:", y_col)

# Δείξε τις πρώτες γραμμές του DataFrame για να επιβεβαιώσεις ότι φορτώθηκε σωστά.
display(df.head())



Shape X, y: (284807, 30) (284807,)
Target column: Class


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [17]:
# ---- SPLIT ----
# Διαχωρισμός σε training και test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,        # ποσοστό του test set (π.χ. 30%)
    random_state=RANDOM_STATE,  # σπόρος για reproducibility
    stratify=y                  # διατήρηση αναλογίας 0/1 (fraud vs non-fraud)
)

# Έλεγχος της ισορροπίας κλάσεων στο test set (σε %)
y_test.value_counts(normalize=True).rename('test class balance (%)') * 100

Class
0    99.826785
1     0.173215
Name: test class balance (%), dtype: float64


## 3) Grid σε C & Αξιολόγηση @ threshold=0.5

- Για κάθε C φτιάχνουμε Pipeline: **StandardScaler → LogisticRegression(L2, class_weight="balanced")**  
- Υπολογίζουμε metrics στο **0.5** και γράφουμε **ROC/PR** καμπύλες σε `images/week3/`.


«Grid σε C» σημαίνει ότι δοκιμάζουμε πολλαπλές τιμές της υπερπαραμέτρου C στη Logistic Regression, για να βρούμε ποια δίνει την καλύτερη απόδοση (με βάση τη μετρική που μας νοιάζει, π.χ. PR-AUC). Είναι ένα grid search αλλά μόνο πάνω στην παράμετρο C.

Τι είναι το C στη Logistic Regression;
- Η Logistic με penalty='l2' (default) λύνει μια βελτιστοποίηση που τιμωρεί μεγάλα βάρη.
- Το C είναι ο αντίστροφος της έντασης κανονικοποίησης (regularization strength):
    - Μικρό C → Ισχυρή κανονικοποίηση → τα βάρη «μαζεύουν» (πιο ομαλό/απλό μοντέλο) → λιγότερο overfitting, αλλά κινδυνεύει από underfitting.
    - Μεγάλο C → Αδύναμη κανονικοποίηση → τα βάρη μπορούν να «φουσκώσουν» για να ταιριάξουν καλύτερα τα δεδομένα → λιγότερο bias, αλλά κίνδυνος overfitting.
- Ισοδύναμα, αν θες να σκέφτεσαι με λήμματα: συχνά βλέπεις όρο λήμματος λ όπου λ = 1/C. Άρα μικρό C = μεγάλο λ (πολύ regularization).

Γιατί κάνουμε grid σε C;
- Γιατί το «σωστό» επίπεδο regularization εξαρτάται από τα δεδομένα.
- Δοκιμάζουμε τυπικά λογαριθμική σκάλα τιμών (π.χ. 0.001, 0.01, 0.1, 1, 10, 100) και κρατάμε αυτή που μεγιστοποιεί τη μετρική-στόχο:
    - Σε imbalanced προβλήματα (fraud), προτείνεται PR-AUC (Average Precision) ή/και Recall σε ενδιαφέρον threshold.
    - ROC-AUC είναι χρήσιμο, αλλά το PR-AUC συνήθως λέει περισσότερα όταν το θετικό είναι σπάνιο.

Πώς το κάνουμε σωστά
- Pipeline: StandardScaler() → LogisticRegression(class_weight="balanced") για να:
    - κλιμακώσεις features (απαραίτητο σε γραμμικά μοντέλα με regularization),
    - ζυγίσεις τη σπάνια κλάση (balanced).
- Στρατηγική αξιολόγησης:
    - Απλό split (train/valid ή train/test), ή
    - Stratified k-fold cross-validation για πιο σταθερά αποτελέσματα.
- Επιλογή solver: lbfgs είναι καλός για L2. Για L1 θέλεις liblinear ή saga.
- Κλίμακα τιμών: ξεκίνα με np.logspace(-3, 2, 6) → 0.001..100 και στένεψε γύρω από το καλύτερο.

Διαισθητικά τι αλλάζει όταν αλλάζω C;
- Μικρό C: τα coefficients μικραίνουν, τα decision boundaries γίνονται πιο «συντηρητικά». Συνήθως μικρό variance, πιθανό μεγαλύτερο bias.
- Μεγάλο C: το μοντέλο «κυνηγά» την εκπαίδευση πιο επιθετικά. Πιθανό overfitting, ειδικά αν έχεις πολλά/συγγραμμικά features.

In [ ]:

# Συλλογή αποτελεσμάτων για report & επιλογή καλύτερου C
rows = []                 # θα κρατήσουμε εδώ τις μετρικές για κάθε C (μία γραμμή ανά C)
best_key = None           # το καλύτερο C βάσει PR-AUC (Average Precision)
best_score = -np.inf      # αρχικοποίηση "χειρότερου" σκορ για σύγκριση
proba_per_C = {}          # πιθανότητες y_proba για το "best C" (για threshold tuning αργότερα)

# rows → θα γίνει DataFrame για γρήγορη σύγκριση/αποθήκευση σε CSV.
# best_score → ξεκινάς από “πολύ μικρό” για να πιάσει ο πρώτος υποψήφιος.
# proba_per_C → αποθηκεύεις για όλα τα C ώστε μετά να μπορείς να ξαναχρησιμοποιήσεις/συγκρίνεις.

# Βρόχος στο grid CS
for C in CS:
    # Pipeline: κανονικοποίηση χαρακτηριστικών + Logistic Regression με class_weight="balanced"
    pipe = Pipeline([
        ("scaler", StandardScaler()),  # Standardization: απαραίτητο για γραμμικά μοντέλα με L2
        ("clf", LogisticRegression(
            max_iter=1000,            # περισσότερα βήματα για σιγουριά σύγκλισης
            class_weight="balanced",  # ζύγισε τη σπάνια κλάση (fraud) για imbalanced data
            penalty="l2",             # default L2 regularization (Ridge-like)
            C=C,                      # υπερπαράμετρος regularization: μικρό C => πιο ισχυρό reg.
            solver="lbfgs",           # σταθερός solver για L2
            random_state=RANDOM_STATE # reproducibility στο optimization
        ))
    ])

    # Εκπαίδευση στο training set
    pipe.fit(X_train, y_train)

# Pipeline αποφεύγει data leakage (το StandardScaler γίνεται fit μόνο στο train και transform στο test).
# class_weight="balanced" → πολύ σημαντικό σε imbalanced (π.χ. fraud) για να μη «αγνοείται» η θετική κλάση.
# C (αντίστροφο της regularization): μικρό C ⇒ πιο «σφιχτή» L2, μεγάλο C ⇒ πιο «χαλαρή».
# solver="lbfgs": σταθερός για L2.
# random_state: με lbfgs ουσιαστικά δεν έχει επίδραση (χρησιμοποιείται σε liblinear/saga), αλλά δεν βλάπτει να υπάρχει για ομοιομορφία API.

    # Πιθανότητες για θετική κλάση (fraud) στο test
    proba = pipe.predict_proba(X_test)[:, 1]

    # Προβλέψεις με default threshold 0.5 (baseline για metrics που θέλουν labels)
    pred_05 = (proba >= 0.5).astype(int)

# Για metrics που είναι threshold-free (ROC-AUC, AP), χρησιμοποιείς probabilities (proba).
# Για metrics που θέλουν labels (accuracy, precision, recall, f1), χρειάζεσαι ένα threshold (baseline: 0.5).

    # Μετρικές απόδοσης
    acc    = accuracy_score(y_test, pred_05)                         # προσοχή: παραπλανητικό σε imbalanced
    prec   = precision_score(y_test, pred_05, zero_division=0)       # zero_division=0: ασφαλές σε corner cases
    rec    = recall_score(y_test, pred_05, zero_division=0)
    f1     = f1_score(y_test, pred_05, zero_division=0)
    rocauc = roc_auc_score(y_test, proba)                            # threshold-free, με βάση τις πιθανότητες
    ap     = average_precision_score(y_test, proba)                  # PR-AUC (προτιμητέο σε imbalanced)

# Accuracy: σε imbalanced datasets είναι συχνά παραπλανητικό.
# Precision/Recall/F1: εξαρτώνται από το threshold (εδώ 0.5).
# ROC-AUC: «γενική» ικανότητα διάκρισης (TPR vs FPR) ανεξάρτητα από threshold.
# Average Precision (PR-AUC): πιο κατάλληλη όταν η θετική κλάση είναι σπάνια.
# ➜ Γι’ αυτό το χρησιμοποιείς ως primary metric για επιλογή C.
# zero_division=0 → αν (ακραία) δεν προβλέφθηκαν θετικά, δεν θα σκάσει exception.

    # Αποθήκευση γραμμής μετρικών (στρογγυλεμένες για καθαρή εκτύπωση/CSV)
    rows.append({
        "C": C,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "ROC-AUC": round(rocauc, 4),
        "PR-AUC(AP)": round(ap, 4)
    })

# Καλό για καθαρό report (στρογγυλοποίηση) και CSV export.
# Πρόταση: κράτα κι ένα n_pos_pred = pred_05.sum() (πόσα θετικά προέβλεψε) — βοηθά στην ανάγνωση trade-offs.

    # --- Αποθήκευση καμπυλών (ROC & PR) σε αρχεία PNG για το report ---
    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.figure()
    plt.plot(fpr, tpr, label=f"ROC-AUC={rocauc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")  # γραμμή τυχαίας πρόβλεψης
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve (C={C})"); plt.legend()
    plt.savefig(outdir / f"roc_C{C}.png", dpi=150, bbox_inches="tight"); plt.close()

# Χρήσιμο για οπτική σύγκριση ανά C.
# plt.close() για να μη σωρεύονται figures στη μνήμη.
# Ονοματοδοσία ανά C → καθαρό repo/report.

    # Precision–Recall curve
    pr, rc, _ = precision_recall_curve(y_test, proba)
    plt.figure()
    plt.plot(rc, pr, label=f"AP={ap:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve (C={C})"); plt.legend()
    plt.savefig(outdir / f"pr_C{C}.png", dpi=150, bbox_inches="tight"); plt.close()

    # --- Επιλογή καλύτερου C βάσει PR-AUC (AP) ---
    if ap > best_score:
        best_score = ap
        best_key = C
        proba_per_C[C] = proba   # αποθήκευσε τις πιθανότητες του καλύτερου C για threshold tuning
                                 # (παραμένουν οι πιο πρόσφατες, δηλ. του τρέχοντος "best")
        
# Εκτύπωση πίνακα μετρικών για όλες τις τιμές του C (baseline threshold = 0.5)
print_metrics_table(rows, title="Metrics per C @ threshold=0.5")

# Αναφορά του καλύτερου C με βάση PR-AUC
print("\n>>> Best C by PR-AUC(AP):", best_key)


C,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC(AP)
0.010000,0.979200,0.068700,0.878400,0.127500,0.968900,0.711400
0.100000,0.978700,0.067300,0.878400,0.124900,0.968200,0.697900
1.000000,0.978600,0.067100,0.878400,0.124700,0.968100,0.705000
10.000000,0.978700,0.067300,0.878400,0.124900,0.968200,0.704900



>>> Best C by PR-AUC(AP): 0.01



## 4) Threshold Tuning (στόχος: Recall ≥ 0.90)

- Επιλέγουμε **threshold** στο **best C** με συνθήκη `Recall ≥ RECALL_TARGET`.  
- Αναφέρουμε metrics στο νέο threshold & αποθηκεύουμε PR curve με σημείο threshold + Confusion Matrix.


In [20]:

# -------------------- Threshold tuning στο καλύτερο C --------------------

best_C = best_key
proba_best = proba_per_C[best_C]  # πιθανότητες (κλάση 1) στο test για το best C

# Βρες threshold που πιάνει στόχο Recall (και όσο γίνεται καλύτερο Precision)
thr, p_at_thr, r_at_thr = choose_threshold_by_recall(
    y_test, proba_best, recall_target=RECALL_TARGET
)
pred_thr = (proba_best >= thr).astype(int)  # labels με custom threshold

# Μετρικές στο επιλεγμένο threshold
acc    = accuracy_score(y_test, pred_thr)
prec   = precision_score(y_test, pred_thr, zero_division=0)
rec    = recall_score(y_test, pred_thr, zero_division=0)
f1     = f1_score(y_test, pred_thr, zero_division=0)
rocauc = roc_auc_score(y_test, proba_best)          # threshold-free
ap     = average_precision_score(y_test, proba_best) # PR-AUC

print(f"Selected threshold = {thr:.4f}")
print(
    f"Metrics @thr: Accuracy={acc:.4f}  Precision={prec:.4f}  "
    f"Recall={rec:.4f}  F1={f1:.4f}  ROC-AUC={rocauc:.4f}  PR-AUC(AP)={ap:.4f}"
)

# -------------------- PR curve με σημείωση του threshold --------------------
pr, rc, thresholds = precision_recall_curve(y_test, proba_best)
plt.figure()
plt.plot(rc, pr, label=f"AP={ap:.3f}")

# Σημείωσε το πιο κοντινό σημείο στο επιλεγμένο threshold (αν υπάρχουν thresholds)
if len(thresholds) > 0:
    idx = int(np.argmin(np.abs(thresholds - thr)))
    plt.scatter(rc[idx], pr[idx], marker="o")
    plt.annotate(
        f"thr={thr:.3f}\nP={pr[idx]:.2f}, R={rc[idx]:.2f}",
        (rc[idx], pr[idx]),
        textcoords="offset points",
        xytext=(10, -10)
    )

plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title(f"Precision–Recall Curve (best C={best_C})")
plt.legend()
plt.savefig(outdir / f"pr_bestC{best_C}_thr.png", dpi=150, bbox_inches="tight"); plt.close()

# -------------------- Confusion Matrix (και counts) --------------------
cm = confusion_matrix(y_test, pred_thr)
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (None, None, None, None)
print(f"Confusion matrix @thr={thr:.3f}:\n{cm}")
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")

# Απλό heatmap-like χωρίς seaborn
plt.figure()
plt.imshow(cm, interpolation="nearest")
plt.title(f"Confusion Matrix @thr={thr:.3f} (C={best_C})")
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ["Pred 0", "Pred 1"])
plt.yticks(tick_marks, ["True 0", "True 1"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout()
plt.xlabel("Predicted label"); plt.ylabel("True label")
plt.savefig(outdir / f"cm_bestC{best_C}_thr.png", dpi=150, bbox_inches="tight"); plt.close()


Selected threshold = 0.2259
Metrics @thr: Accuracy=0.9302  Precision=0.0220  Recall=0.9054  F1=0.0430  ROC-AUC=0.9689  PR-AUC(AP)=0.7114
Confusion matrix @thr=0.226:
[[79349  5946]
 [   14   134]]
TP=134  FP=5946  FN=14  TN=79349


## 5) Καταγραφή Αποτελεσμάτων & Commit

### Best Model (Grid σε C)
- Επιλεγμένο **C**: `0.01`
- Βασικό metric επιλογής: **PR-AUC (Average Precision)** = `0.7114`
- ROC-AUC = `0.9689`

### Threshold Tuning
- Επιλεγμένο **Threshold**: `0.2259`
- Μετρικές στο threshold:
  - Accuracy = `0.9302`
  - Precision = `0.0220`
  - Recall = `0.9054` ✅ (πιάσαμε στόχο ≥ 0.90)
  - F1 = `0.0430`
- Confusion Matrix:
[[79349 5946]
[ 14 134]]
- TN = 79,349  
- FP = 5,946  
- FN = 14  
- TP = 134  

- **Trade-offs:**  
- Αυξημένο Recall ⇒ εντοπίζουμε το 90.5% των fraud συναλλαγών.  
- Όμως η Precision είναι πολύ χαμηλή (2.2%) ⇒ πολλά False Positives (~5.9k περιπτώσεις).  
- Σε fraud detection αυτό είναι αποδεκτό: προτιμάμε να μην χάσουμε ένα fraud (FN) ακόμα κι αν σημαίνει πολλά alerts για έλεγχο.


---

### Tips
- **StandardScaler** πριν από L2 logistic regression για σταθερότητα/σύγκλιση.
- **Μικρό C ⇒ πιο ισχυρή L2** (καλό όταν βλέπεις overfitting).
- Σε **fraud detection** το **Recall** είναι συχνά προτεραιότητα· έχε επίγνωση του κόστους από **FP**.
- Για πιο σταθερή εκτίμηση, προχώρα σε **Stratified K-Fold CV** & **learning curves** (Ημέρα 3–4).
